# 06｜注意力寻宝综合项目 🧭

你要调查 Transformer 怎样从一串数量相同的颜色中找回第 0 位，并在固定资源预算内完成一次代码修改和四枚徽章挑战。

| 阶段 | 建议时间 | 必交证据 |
|---|---:|---|
| 基线与任务预测 | 20 分钟 | 基线结果、检索位置预测 |
| Token、Embedding 与位置分析 | 35 分钟 | shape 表、位置编码图 |
| Q/K/V 与 Attention 取证 | 40 分钟 | 一次手算、QUERY 权重解释 |
| 三组单变量对照 | 55 分钟 | 公平对照记录 |
| 阅读并修改 `model.py` | 45 分钟 | 一处最小修改、烟雾测试 |
| 预算内挑战 | 25 分钟 | 最多 3 次尝试、最佳配置 |
| 证据报告 | 20 分钟 | 300–500 字有限结论 |
| **合计** | **240 分钟** | **约 4 小时** |

四小时来自观察、手算、编程和表达，不来自长时间占用服务器。

In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import torch

LAB_DIR = Path.cwd()
if not (LAB_DIR / "transformer_lab").is_dir():
    LAB_DIR = (LAB_DIR / "04_transformer_lab").resolve()
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

torch.set_num_threads(1)

from dataclasses import replace
from transformer_lab import (
    ExperimentConfig, challenge_report, make_retrieval_data,
    plot_attention_weights, plot_model_attention, plot_positional_encoding,
    plot_tensor_structure, plot_token_sequence, print_result_table,
    run_experiment, scaled_dot_product_attention, validate_config,
)


## 项目规则

- 固定使用 `DEVICE = "cpu"`，不占课堂 GPU；
- 全项目最多 8 次正式训练：基线 1 次、单变量对照 3 次、代码修改 1 次、Boss 最多 3 次；
- 每组实验运行前必须写预测，一次只改一个变量；
- 评分不使用运行时间，也不能靠增加 Epoch 刷分；
- 没集齐徽章不等于失败；预测、修改、证据和边界完整即可。

## 热身｜基线与四徽章（20 分钟）

先预测 QUERY 会关注哪个位置，再运行一层 Transformer 基线。

In [ ]:
DEVICE = "cpu"
baseline_config = ExperimentConfig(
    name="综合项目基线",
    content_length=12,
    d_model=32,
    num_heads=4,
    num_layers=1,
    dim_feedforward=64,
    dropout=0.1,
    activation="gelu",
    epochs=12,
    seed=42,
    device=DEVICE,
)
baseline = run_experiment(baseline_config)
print_result_table([baseline])

test_x, test_y = next(iter(baseline["loaders"]["test"]))
plot_token_sequence(test_x, test_y)
plot_model_attention(baseline["model"], test_x, device=DEVICE)
baseline_score = challenge_report(baseline)


### 热身记录

基线得到 __/4 枚徽章。QUERY 权重最高的位置是 __。这是否足以证明模型“只使用了该位置”？__。

## 第一关｜Token 与位置（35 分钟）

找出 Batch、Time、Embedding 三个维度，并验证颜色计数相同、三个数据集无精确序列重叠。

In [ ]:
data = baseline["data"]
plot_tensor_structure(data)
plot_positional_encoding(d_model=32, max_length=25)

sample = data.x_train[0]
counts = [(sample[:-1] == token).sum().item() for token in range(3)]
as_rows = lambda tensor: {tuple(row.tolist()) for row in tensor}
train_rows, val_rows, test_rows = map(
    as_rows, (data.x_train, data.x_val, data.x_test)
)
print("红/绿/蓝计数：", counts)
print("训练∩验证、训练∩测试、验证∩测试：",
      len(train_rows & val_rows), len(train_rows & test_rows), len(val_rows & test_rows))
print("token ids [N,T]：", tuple(data.x_train.shape))
print("Embedding [B,T,D]：", tuple(baseline["model"].embedding(data.x_train[:8]).shape))


### Shape 与位置证词

| 对象 | Shape | 每一维含义 |
|---|---|---|
| token ids |  |  |
| Embedding |  |  |
| Attention |  |  |
| logits |  |  |

如果关闭位置编码，我预测 __，因为 __。

## 第二关｜Q/K/V 与 Attention 取证（40 分钟）

先手算一个 Query 的点积、缩放、Softmax 和 Value 加权，再与模型的 QUERY Attention 图联系起来。

In [ ]:
query = torch.tensor([1.0, 0.0])
keys = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
values = torch.tensor([[2.0, 0.0], [0.0, 4.0], [1.0, 1.0]])

scores, weights, output = scaled_dot_product_attention(query, keys, values)
print("scores：", scores.numpy().round(3).tolist())
print("weights：", weights.numpy().round(3).tolist(), "｜和 =", float(weights.sum()))
print("加权输出：", output.numpy().round(3).tolist())
plot_attention_weights(weights, ["位置0", "位置1", "位置2"], "手算 Attention")
plot_model_attention(baseline["model"], test_x[:1], device=DEVICE)


### Attention 证词

手算中位置 __ 权重最高，因为 __。基线 QUERY 图中位置 __ 权重较高。它支持的有限表述是 __；它不能证明 __。

## 第三关｜三组单变量对照（55 分钟）

三组分别关闭位置编码、把 Head 改为 1、把 Dropout 改为 0。其他条件保持与基线相同。运行前填写预测。

In [ ]:
TRIALS = {
    "关闭位置编码": {"use_positional_encoding": False},
    "Head 4→1": {"num_heads": 1},
    "Dropout 0.1→0": {"dropout": 0.0},
}

trial_results = {}
for label, one_change in TRIALS.items():
    print(f"\n===== {label}：{one_change} =====")
    config = replace(baseline_config, name=label, **one_change)
    trial_results[label] = run_experiment(config)
    print_result_table([trial_results[label]])


### 对照记录

| 实验 | 运行前预测 | 唯一修改 | 测试 Accuracy | 参数量 | Attention 预算 | 有边界解释 |
|---|---|---|---:|---:|---:|---|
| 关闭位置编码 |  |  |  |  |  |  |
| Head 4→1 |  |  |  |  |  |  |
| Dropout 0.1→0 |  |  |  |  |  |  |

不能把一次实验写成“Head 越多越好”或“Dropout 一定提高 Accuracy”。

## 安全门演示｜拒绝无效工作量

下面故意提交 100 个 Epoch。程序应在生成数据和分配模型前拒绝，不会开始训练。

In [ ]:
unsafe = replace(baseline_config, epochs=100)
try:
    validate_config(unsafe)
except ValueError as error:
    print("安全门已拦截：", error)


## 第四关｜阅读并修改模型（45 分钟）

1. 打开 `transformer_lab/model.py`，画出 Embedding → Position → Attention → Add & Norm → Feed-forward → Add & Norm → Classifier；
2. 在 `_activation()` 的 `choices` 中增加 PyTorch 已有激活函数，例如 `"silu": nn.SiLU()`；
3. 不复制 Encoder、训练循环或新建依赖；
4. 将下方 `ACTIVATION_TO_TEST` 换成新增名称；
5. 运行 `python test_transformer_smoke.py` 并保存通过截图。

In [ ]:
# 修改 model.py 前使用 relu 可直接运行；完成后换成你新增的名称。
ACTIVATION_TO_TEST = "relu"
code_config = replace(
    baseline_config,
    name=f"代码修改：{ACTIVATION_TO_TEST}",
    activation=ACTIVATION_TO_TEST,
)
code_result = run_experiment(code_config)
print_result_table([baseline, code_result])


### 代码修改记录

修改文件与行：__。新增映射：__。为什么不需要重写 Attention 或训练循环：__。

烟雾测试结果：__。与基线相比，参数量 __，测试 Accuracy __。

## Boss 关｜三次机会争取四徽章（25 分钟）

目标：测试 Accuracy ≥85%、长度 24 迁移 ≥45%、参数量 ≤15000、Attention 组合预算 ≤3000000。

最多尝试 3 个配置，每次先写预测。可修改 Head、Feed-forward 宽度、Dropout、激活函数或学习率；保持内容长度 12、Epoch≤12、seed=42。

In [ ]:
BOSS_CHANGES = dict(
    num_heads=4,
    dim_feedforward=64,
    dropout=0.1,
    activation="gelu",
    learning_rate=0.003,
)

boss_config = replace(
    baseline_config,
    name="Boss 尝试",
    **BOSS_CHANGES,
)
boss_result = run_experiment(boss_config)
print_result_table([boss_result])
boss_score = challenge_report(boss_result)


### Boss 尝试记录

| 次数 | 运行前预测 | 唯一/主要修改 | 四项数值 | 徽章数 | 下一步 |
|---:|---|---|---|---:|---|
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |
| 3 |  |  |  |  |  |

没有集齐四徽章时，说明最可能的瓶颈和下一步即可；不要突破安全上限。

## 最终提交｜300–500 字证据报告（20 分钟）

按顺序写：

1. 任务问题与运行前预测；
2. 数据为何不能靠颜色计数作弊；
3. 三组对照中保持不变的条件和唯一修改；
4. 至少两项数值证据，其中一项必须是长度迁移、参数量或 Attention 预算；
5. 一项具体图形证据；
6. `model.py` 修改位置与烟雾测试；
7. 使用“在当前合成数据、seed、序列长度和训练预算下”限定结论。

提交：完成后的 Notebook、最小代码修改、测试截图和证据报告。

## 完成后：保存并释放资源

先保存 Notebook，再运行最后一格。

In [ ]:
%reset -f
import gc
import matplotlib.pyplot as plt
import torch
from IPython import get_ipython

plt.close("all")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("注意力寻宝记录已保存，正在结束当前 Notebook 内核……")
get_ipython().kernel.do_shutdown(restart=False)
